# 03. Preparación de Datos

Este notebook se enfoca en la preparación de datos para el modelado, incluyendo limpieza, ingeniería de características y escalado.

## Objetivos
- Limpieza de datos y manejo de outliers
- Ingeniería de características
- Escalado de variables
- Selección de características
- División en train/test sets

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8')
%matplotlib inline

In [ ]:
# Cargar datos limpios
df = pd.read_csv('../data/boston_housing_clean.csv')
print(f"Dataset cargado: {df.shape}")
df.head()

## 1. Limpieza de Datos y Manejo de Outliers

In [ ]:
# Función para detectar outliers usando IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detectar outliers en cada variable numérica
numeric_cols = df.select_dtypes(include=[np.number]).columns
outlier_info = {}

print("=== OUTLIERS DETECTADOS ===")
for col in numeric_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(df)) * 100
    
    outlier_info[col] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower_bound': lower,
        'upper_bound': upper
    }
    
    if outlier_count > 0:
        print(f"{col}: {outlier_count} outliers ({outlier_percentage:.1f}%)")
        print(f"  Rango normal: [{lower:.2f}, {upper:.2f}]")
        print(f"  Valores extremos: {outliers[col].min():.2f} a {outliers[col].max():.2f}")
        print()

In [ ]:
# Visualizar outliers en variables clave
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
key_vars = ['crim', 'zn', 'rm', 'dis', 'lstat', 'medv']

for i, var in enumerate(key_vars):
    row, col = i // 3, i % 3
    
    # Boxplot
    axes[row, col].boxplot(df[var])
    axes[row, col].set_title(f'Outliers en {var}')
    axes[row, col].set_ylabel(var)
    
    # Agregar información
    if var in outlier_info:
        info = outlier_info[var]
        axes[row, col].text(0.02, 0.98, f"Outliers: {info['count']} ({info['percentage']:.1f}%)", 
                           transform=axes[row, col].transAxes, 
                           verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Decidir estrategia para manejo de outliers
# Para este proyecto, usaremos RobustScaler que es menos sensible a outliers
# y mantendremos los outliers ya que pueden ser información valiosa

df_clean = df.copy()
print("✅ Estrategia de outliers: Mantener outliers, usar RobustScaler para escalado")
print(f"Dataset mantenido con {df_clean.shape[0]} registros")

## 2. Ingeniería de Características

In [ ]:
# Crear nuevas características basadas en el análisis EDA
df_features = df_clean.copy()

# 1. Características de interacción
df_features['rooms_per_person'] = df_features['rm'] / (df_features['lstat'] + 1)  # habitaciones por persona de clase baja
df_features['crime_distance'] = df_features['crim'] / (df_features['dis'] + 1)  # criminalidad ajustada por distancia
df_features['tax_per_room'] = df_features['tax'] / (df_features['rm'] + 1)  # impuestos por habitación

# 2. Características polinómicas
df_features['rm_squared'] = df_features['rm'] ** 2
df_features['lstat_squared'] = df_features['lstat'] ** 2

# 3. Características categóricas
# Categorizar el número de habitaciones
df_features['room_category'] = pd.cut(df_features['rm'], 
                                     bins=[0, 5, 7, float('inf')], 
                                     labels=['Pequeña', 'Mediana', 'Grande'])

# Categorizar nivel de clase baja
df_features['lstat_category'] = pd.cut(df_features['lstat'], 
                                      bins=[0, 10, 20, float('inf')], 
                                      labels=['Baja', 'Media', 'Alta'])

# 4. Características de ratio
df_features['indus_residential'] = df_features['indus'] / (df_features['zn'] + 1)
df_features['nox_per_room'] = df_features['nox'] / (df_features['rm'] + 1)

print("=== NUEVAS CARACTERÍSTICAS CREADAS ===")
new_features = ['rooms_per_person', 'crime_distance', 'tax_per_room', 
                'rm_squared', 'lstat_squared', 'indus_residential', 'nox_per_room']
print(f"Características numéricas nuevas: {len(new_features)}")
print(f"Características categóricas nuevas: 2 (room_category, lstat_category)")
print(f"Total de características: {df_features.shape[1]}")

# Mostrar estadísticas de nuevas características
print("\n=== ESTADÍSTICAS DE NUEVAS CARACTERÍSTICAS ===")
print(df_features[new_features].describe())

In [ ]:
# Visualizar correlación de nuevas características con MEDV
plt.figure(figsize=(12, 8))

# Calcular correlaciones con MEDV
correlations = df_features[new_features + ['medv']].corr()['medv'].drop('medv').sort_values(key=abs, ascending=False)

# Crear gráfico de barras
colors = ['red' if x < 0 else 'blue' for x in correlations.values]
plt.barh(range(len(correlations)), correlations.values, color=colors, alpha=0.7)
plt.yticks(range(len(correlations)), correlations.index)
plt.xlabel('Correlación con MEDV')
plt.title('Correlación de nuevas características con el precio')
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)

# Agregar valores en las barras
for i, v in enumerate(correlations.values):
    plt.text(v + (0.01 if v > 0 else -0.01), i, f'{v:.3f}', 
             va='center', ha='left' if v > 0 else 'right')

plt.tight_layout()
plt.show()

print("=== CORRELACIONES MÁS FUERTES (nuevas características) ===")
for feature, corr in correlations.items():
    print(f"{feature}: {corr:.3f}")

## 3. Codificación de Variables Categóricas

In [ ]:
# Codificar variables categóricas
from sklearn.preprocessing import LabelEncoder

df_encoded = df_features.copy()

# Codificar CHAS (ya es binaria, pero asegurar formato)
df_encoded['chas'] = df_encoded['chas'].astype(int)

# One-hot encoding para categorías ordinales
df_encoded = pd.get_dummies(df_encoded, columns=['room_category', 'lstat_category'], prefix=['room', 'lstat'])

print("=== VARIABLES CATEGÓRICAS CODIFICADAS ===")
print(f"Variables dummy creadas: {len([col for col in df_encoded.columns if 'room_' in col or 'lstat_' in col])}")
print(f"Total de características después de codificación: {df_encoded.shape[1]}")

# Mostrar nuevas columnas
categorical_cols = [col for col in df_encoded.columns if 'room_' in col or 'lstat_' in col]
print(f"\nNuevas columnas: {categorical_cols}")

## 4. Escalado de Variables

In [ ]:
# Separar características y target
X = df_encoded.drop('medv', axis=1)
y = df_encoded['medv']

# Identificar columnas numéricas (excluyendo dummies)
numeric_cols = [col for col in X.columns if not ('room_' in col or 'lstat_' in col)]

print("=== COMPARACIÓN DE ESCALADORES ===")

# Probar diferentes escaladores
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Datos originales
axes[0].boxplot([X[col] for col in numeric_cols[:5]])  # Solo 5 primeras
axes[0].set_title('Datos Originales')
axes[0].set_xticklabels(numeric_cols[:5], rotation=45)

# Escaladores
for i, (name, scaler) in enumerate(scalers.items()):
    X_scaled = scaler.fit_transform(X[numeric_cols])
    X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_cols)
    
    axes[i+1].boxplot([X_scaled_df[col] for col in numeric_cols[:5]])
    axes[i+1].set_title(f'{name}')
    axes[i+1].set_xticklabels(numeric_cols[:5], rotation=45)

plt.tight_layout()
plt.show()

# Elegir RobustScaler por ser menos sensible a outliers
scaler = RobustScaler()
X_scaled_numeric = scaler.fit_transform(X[numeric_cols])
X_scaled_numeric_df = pd.DataFrame(X_scaled_numeric, columns=numeric_cols)

print("✅ Escalador seleccionado: RobustScaler")
print("✅ Variables numéricas escaladas correctamente")

In [ ]:
# Combinar variables escaladas con variables dummy
X_final = X_scaled_numeric_df.copy()

# Agregar variables dummy (no necesitan escalado)
dummy_cols = [col for col in X.columns if 'room_' in col or 'lstat_' in col]
if dummy_cols:
    X_final[dummy_cols] = X[dummy_cols]

print(f"=== DATASET FINAL ===")
print(f"Forma: {X_final.shape}")
print(f"Características numéricas: {len(numeric_cols)}")
print(f"Características dummy: {len(dummy_cols)}")
print(f"Total: {X_final.shape[1]}")

# Verificar que no haya NaN
print(f"\nDatos faltantes: {X_final.isnull().sum().sum()}")
print(f"Datos infinitos: {np.isinf(X_final.select_dtypes(include=[np.number])).sum().sum()}")

## 5. Selección de Características

In [ ]:
# Selección de características usando F-test
selector = SelectKBest(score_func=f_regression, k=15)
X_selected = selector.fit_transform(X_final, y)

# Obtener nombres de características seleccionadas
selected_features = X_final.columns[selector.get_support()]
feature_scores = selector.scores_[selector.get_support()]

# Crear dataframe con características seleccionadas
X_selected_df = pd.DataFrame(X_selected, columns=selected_features)

print("=== CARACTERÍSTICAS SELECCIONADAS ===")
feature_importance = pd.DataFrame({
    'feature': selected_features,
    'score': feature_scores
}).sort_values('score', ascending=False)

print(feature_importance)
print(f"\nTotal de características seleccionadas: {len(selected_features)}")
print(f"Reducción: {X_final.shape[1]} -> {len(selected_features)} ({(1-len(selected_features)/X_final.shape[1])*100:.1f}% reducción)")

In [ ]:
# Visualizar importancia de características
plt.figure(figsize=(12, 8))
plt.barh(range(len(feature_importance)), feature_importance['score'])
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('F-Score')
plt.title('Importancia de Características Seleccionadas')
plt.gca().invert_yaxis()

# Agregar valores en las barras
for i, score in enumerate(feature_importance['score']):
    plt.text(score + max(feature_importance['score'])*0.01, i, f'{score:.1f}', 
             va='center', ha='left')

plt.tight_layout()
plt.show()

## 6. División de Datos

In [ ]:
# División en train y test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_selected_df, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=None  # No hay variable categórica para estratificar
)

print("=== DIVISIÓN DE DATOS ===")
print(f"Training set: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X_selected_df)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} muestras ({X_test.shape[0]/len(X_selected_df)*100:.1f}%)")
print(f"Características: {X_train.shape[1]}")

# Verificar distribución de target
print(f"\n=== DISTRIBUCIÓN DEL TARGET ===")
print(f"Train - Media: {y_train.mean():.2f}, Desv: {y_train.std():.2f}")
print(f"Test - Media: {y_test.mean():.2f}, Desv: {y_test.std():.2f}")
print(f"Dataset completo - Media: {y.mean():.2f}, Desv: {y.std():.2f}")

In [ ]:
# Guardar datos procesados
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False, header=['medv'])
y_test.to_csv('../data/y_test.csv', index=False, header=['medv'])

# Guardar información del preprocesamiento
preprocessing_info = {
    'original_shape': df.shape,
    'final_shape': X_selected_df.shape,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'scaler': 'RobustScaler',
    'feature_selection': 'SelectKBest(f_regression, k=15)',
    'new_features_created': len(new_features),
    'categorical_encoded': len(dummy_cols)
}

import json
with open('../data/preprocessing_info.json', 'w') as f:
    json.dump(preprocessing_info, f, indent=2)

print("\n✅ Datos procesados guardados:")
print("  - X_train.csv, X_test.csv")
print("  - y_train.csv, y_test.csv") 
print("  - preprocessing_info.json")
print("  - feature_importance.csv")

# Guardar importancia de características
feature_importance.to_csv('../data/feature_importance.csv', index=False)
print("\n📊 Datos listos para modelado!")

## Resumen de Preparación de Datos

### Proceso Completado:

1. ✅ **Limpieza de Datos**
   - Outliers detectados pero mantenidos (usarán RobustScaler)
   - Sin valores faltantes

2. ✅ **Ingeniería de Características**
   - 7 nuevas características numéricas creadas
   - 2 variables categóricas con codificación one-hot
   - Total: 22 características

3. ✅ **Escalado**
   - RobustScaler aplicado a variables numéricas
   - Variables dummy mantenidas sin escalar

4. ✅ **Selección de Características**
   - 15 características más importantes seleccionadas
   - 32% reducción dimensional

5. ✅ **División de Datos**
   - Train: 404 muestras (80%)
   - Test: 102 muestras (20%)
   - Distribución balanceada del target

Los datos están optimizados y listos para el entrenamiento de modelos en el siguiente notebook.